# Task 1: Understanding the Phishing Problem, Dataset Audit & Schema Harmonization
**Course:** AI in Cyber Security Lab (`ENSP355`) — Unit 1 Capstone  
**Student:** Yatharth Chopra (`2301420022`) | **Program:** B.Tech CSE (Data Science), Sem 7  
**University:** K.R. Mangalam University | **Faculty:** Monika Khatkar

---

### Understanding the Problem Behind Phishing Detection
Imagine you are a Security Operations Center (SOC) analyst protecting an enterprise network. Every day, employees click thousands of URLs:

```text
Employee Browser  ---------->  Legitimate Portal (Valid SSL, Established Domain)
Employee Browser  ---------->  Phishing Spoof    (Missing/Fake SSL, External Anchors, IP-in-URL)
```

Static blocklists only stop **already-reported** domains. Attackers bypass them in minutes by spinning up fresh domains.  
**The central question becomes:**  
> *Can we use structural URL, SSL/TLS, and HTML indicators to automatically distinguish legitimate websites from phishing threats—even on sites never seen before?*

### Objectives of Task 1
1. Validate the Python ML/DL environment (`numpy`, `pandas`, `scikit-learn`, `torch`, `matplotlib`, `seaborn`, `reportlab`).
2. Audit both raw UCI phishing datasets (`UCI 327` and `UCI 379`) for shape, missing values, and duplicates.
3. Perform a **row-wise union** on the **9 shared attributes**, detect and fix **feature polarity inversion** in Dataset B, and export `data/merged_phishing.csv`.

## 1) Environment & Dependency Verification

In [1]:
import sys, importlib, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from common import CSV_A, CSV_B, MERGED_FILE, OUT_DIR, FEATURES

print(f"Python Version: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "sklearn", "torch", "matplotlib", "seaborn", "reportlab"]:
    m = importlib.import_module(pkg)
    print(f"  [OK] {pkg:<12} {getattr(m, '__version__', '?')}")

Python Version: 3.14.7
  [OK] numpy        2.4.2
  [OK] pandas       3.0.2


  [OK] sklearn      1.8.0


  [OK] torch        2.10.0+cpu
  [OK] matplotlib   3.10.8


  [OK] seaborn      0.13.2
  [OK] reportlab    5.1.2


## 2) Dataset Audit (Shapes, Head, Missing Values & Duplicates)

In [2]:
df_a = pd.read_csv(CSV_A)
df_b = pd.read_csv(CSV_B)

print(f"Dataset A (UCI 327): {df_a.shape[0]:,} rows x {df_a.shape[1]} cols | Nulls: {int(df_a.isna().sum().sum())} | Duplicates: {int(df_a.duplicated().sum())}")
print(f"Dataset B (UCI 379): {df_b.shape[0]:,} rows x {df_b.shape[1]} cols | Nulls: {int(df_b.isna().sum().sum())} | Duplicates: {int(df_b.duplicated().sum())}")
print("\n9 Shared Features:", FEATURES)
df_a[FEATURES + ["Result"]].head()

Dataset A (UCI 327): 11,055 rows x 31 cols | Nulls: 0 | Duplicates: 5206
Dataset B (UCI 379): 1,353 rows x 10 cols | Nulls: 0 | Duplicates: 629

9 Shared Features: ['SFH', 'popUpWidnow', 'SSLfinal_State', 'Request_URL', 'URL_of_Anchor', 'web_traffic', 'URL_Length', 'age_of_domain', 'having_IP_Address']


,SFH,popUpWidnow,SSLfinal_State,Request_URL,URL_of_Anchor,web_traffic,URL_Length,age_of_domain,having_IP_Address,Result
0,-1,1,-1,1,-1,-1,1,-1,-1,-1
1,-1,1,1,1,0,0,1,-1,1,-1
2,-1,1,-1,1,0,1,0,1,1,-1
3,-1,1,-1,-1,0,1,0,-1,1,-1
4,-1,-1,1,1,0,0,0,-1,1,1


## 3) Schema Harmonisation & Polarity Audit
Although both UCI pages document `+1 = legitimate`, `0 = suspicious`, `-1 = phishy`, inspecting Spearman correlations with `Result` revealed a critical data-engineering issue: **8 of the 9 shared features in Dataset B were coded in the opposite direction** (and `having_IP_Address` used `{0, 1}` instead of `{-1, 1}`). Stacking the raw files without fixing this would feed contradictory signals to every model!

In [3]:
harm = pd.read_csv(OUT_DIR / "harmonization_report.csv")
display(harm)

merged = pd.read_csv(MERGED_FILE)
print(f"Merged Dataset Shape: {merged.shape[0]:,} rows x {merged.shape[1]} columns | Nulls: {int(merged.isna().sum().sum())}")
print("\nClass Distribution (0 = Legitimate, 1 = Threat):")
print(merged["label"].value_counts().rename({0: "Legitimate (0)", 1: "Threat (1)"}))
merged.head()

,feature,corr_A,corr_B_raw,action,corr_B_after,share_minus1_A,share_minus1_B_after
0,SFH,0.219,-0.728,polarity reversed (x -> -x),0.728,0.763,0.567
1,popUpWidnow,0.000,-0.536,polarity reversed (x -> -x),0.536,0.193,0.135
2,SSLfinal_State,0.736,-0.551,polarity reversed (x -> -x),0.551,0.322,0.555
3,Request_URL,0.253,-0.260,polarity reversed (x -> -x),0.260,0.407,0.233
4,URL_of_Anchor,0.701,-0.295,polarity reversed (x -> -x),0.295,0.297,0.426
5,web_traffic,0.365,0.258,kept as is,0.258,0.240,0.325
6,URL_Length,0.053,-0.199,polarity reversed (x -> -x),0.199,0.810,0.265
7,age_of_domain,0.121,-0.243,polarity reversed (x -> -x),0.243,0.469,0.610
8,having_IP_Address,0.094,-0.062,"recoded {0,1} -> {-1,1}, polarity reversed",0.062,0.343,0.115


Merged Dataset Shape: 12,408 rows x 13 columns | Nulls: 0

Class Distribution (0 = Legitimate, 1 = Threat):
label
Legitimate (0)    6705
Threat (1)        5703
Name: count, dtype: int64


,record_id,source,SFH,popUpWidnow,SSLfinal_State,Request_URL,URL_of_Anchor,web_traffic,URL_Length,age_of_domain,having_IP_Address,result_raw,label
0,1,UCI327_PhishingWebsites,-1,1,-1,1,-1,-1,1,-1,-1,-1,1
1,2,UCI327_PhishingWebsites,-1,1,1,1,0,0,1,-1,1,-1,1
2,3,UCI327_PhishingWebsites,-1,1,-1,1,0,1,0,1,1,-1,1
3,4,UCI327_PhishingWebsites,-1,1,-1,-1,0,1,0,-1,1,-1,1
4,5,UCI327_PhishingWebsites,-1,-1,1,1,0,0,0,-1,1,1,0


## 4) Merge Summary & Theoretical Information Ceiling

In [4]:
ms = json.loads((OUT_DIR / "merge_summary.json").read_text())
for k, v in ms.items():
    print(f"{k:<32}: {v}")

rows_total                      : 12408
rows_A                          : 11055
rows_B                          : 1353
shared_features                 : ['SFH', 'popUpWidnow', 'SSLfinal_State', 'Request_URL', 'URL_of_Anchor', 'web_traffic', 'URL_Length', 'age_of_domain', 'having_IP_Address']
features_dropped_from_A         : 21
label_counts                    : {'0': 6705, '1': 5703}
result_raw_counts               : {'-1': 5600, '0': 103, '1': 6705}
missing_values                  : 0
duplicate_rows_feature_label    : 10716
unique_feature_profiles         : 1508
profiles_with_mixed_labels      : 184
lookup_table_ceiling_accuracy   : 0.9368149580915538
profiles_in_both_sources        : 88
